# Clean BG / Fermi Plots From Repo Data

This notebook reproduces the **same figure format** as `clean_plots.ipynb`, but uses the data stored in this repository and the same fitting procedure we have been using here.

Notes:
- The visual labels keep the `p_c` notation to match the original figure style.
- In the repo data, that role is played by `\lambda_q`.


In [1]:
#import Pkg; Pkg.add("JLD2")

In [2]:
using JLD2
using KadanoffBaym
using Statistics
using LaTeXStrings
import PyPlot as plt


LoadError: ArgumentError: Package KadanoffBaym not found in current path.
- Run `import Pkg; Pkg.add("KadanoffBaym")` to install the KadanoffBaym package.

In [ ]:
HAS_LATEX = Sys.which("latex") !== nothing
plt.rc("text", usetex=HAS_LATEX)
plt.rc("font", family="serif")
if HAS_LATEX
    plt.rc("text.latex", preamble=raw"\usepackage{amsmath}")
end
fs = 21
plt.rc("font", size=fs)
plt.rc("axes", labelsize=fs, titlesize=fs)
plt.rc("xtick", labelsize=fs)
plt.rc("ytick", labelsize=fs)
plt.rc("legend", fontsize=fs)

letter = ["("*string(c)*")" for c in 'a':'z'];


In [ ]:
# Resolve the repository root whether the notebook is opened from the repo root
# or from inside Figures_electron_bath.
repo_root = basename(pwd()) == "Figures_electron_bath" ? abspath(joinpath(pwd(), "..")) : pwd()
data_dir = joinpath(repo_root, "Data")

# Default matched pair: same T_bath and alpha, differing in eta and p_c (lambda_q here).
bg_dataset = "L100_Te1.0_Tb0.1_u0.0_γ1.0_dispersion_α2.5_s1.0_ωc10.0_linear_spectral_η0.05_v_b0.2_ωb00.1_power_exp_s_q1.0_λ_q0.5_t050.0_ω03.141592653589793_σ2.0_A0.0_switch0_ti3.0_to20.0_tmax60"
fermi_dataset = "L100_Te1.0_Tb0.1_u0.0_γ1.0_dispersion_α2.5_s1.0_ωc10.0_linear_spectral_η0.5_v_b0.2_ωb00.1_power_exp_s_q1.0_λ_q0.5_t050.0_ω03.141592653589793_σ2.0_A0.0_switch0_ti3.0_to20.0_tmax60"

snapshot_times = [0.0, 20.0, 60.0]


In [ ]:
array_data(x) = hasproperty(x, :data) ? getproperty(x, :data) : x
fermi(ϵ, T, μ) = 1 / (exp((ϵ - μ) / T) + 1)

function load_nk_times(dataset, data_dir)
    GL = array_data(load(joinpath(data_dir, "GL_" * dataset * ".jld2"), "GL"))
    ts_obj = load(joinpath(data_dir, "ts_" * dataset * ".jld2"), "sol")
    ts = hasproperty(ts_obj, :t) ? ts_obj.t : ts_obj
    L = size(GL, 1)
    Nt = size(GL, 2)
    nk_t = zeros(Float64, Nt, L)
    @inbounds for it in 1:Nt
        nk_t[it, :] .= imag.(GL[:, it, it])
    end
    return ts, nk_t
end

function local_fit_rmse(nk, ϵs, T, μ)
    ff = fermi.(ϵs, T, μ)
    return sum((nk .- ff).^2)
end

function best_fermi_fit(nk, ϵs)
    coarse_T = 0.02:0.02:1.50
    coarse_μ = -3.0:0.05:3.0

    best_sse = Inf
    best_T = NaN
    best_μ = NaN
    for T in coarse_T, μ in coarse_μ
        sse = local_fit_rmse(nk, ϵs, T, μ)
        if sse < best_sse
            best_sse = sse
            best_T = T
            best_μ = μ
        end
    end

    fine_T_min = max(0.01, best_T - 0.04)
    fine_T_max = min(1.50, best_T + 0.04)
    fine_μ_min = max(-3.0, best_μ - 0.10)
    fine_μ_max = min(3.0, best_μ + 0.10)

    for T in fine_T_min:0.002:fine_T_max, μ in fine_μ_min:0.01:fine_μ_max
        sse = local_fit_rmse(nk, ϵs, T, μ)
        if sse < best_sse
            best_sse = sse
            best_T = T
            best_μ = μ
        end
    end

    return best_T, best_μ, best_sse
end

function fit_timeseries(ts, nk_t)
    nk = size(nk_t, 2)
    k = collect(LinRange(-pi, pi, nk))
    ϵs = -2 .* cos.(k)
    teff = zeros(Float64, length(ts))
    sse = zeros(Float64, length(ts))
    μeff = zeros(Float64, length(ts))
    @inbounds for it in eachindex(ts)
        teff[it], μeff[it], sse[it] = best_fermi_fit(view(nk_t, it, :), ϵs)
    end
    return k, teff, μeff, sse
end


In [3]:
bg_t, bg_occ = load_nk_times(bg_dataset, data_dir)
fermi_t, fermi_occ = load_nk_times(fermi_dataset, data_dir)

k, bg_teff, bg_μeff, bg_sse = fit_timeseries(bg_t, bg_occ)
_, fermi_teff, fermi_μeff, fermi_sse = fit_timeseries(fermi_t, fermi_occ)

sidx = [argmin(abs.(bg_t .- t)) for t in snapshot_times]
snapshot_actual_times = bg_t[sidx]

# Keep labels in the same visual language as the original notebook.
bg_eta_label = raw"$\rm \eta = 0.05\gamma \qquad p_c=0.5/a $"
fermi_eta_label_1 = raw"$\rm \eta = 0.5\gamma$"
fermi_eta_label_2 = raw"$\rm p_c=0.5/a$"
bg_eta_label_1 = raw"$\rm \eta = 0.05\gamma$"
bg_eta_label_2 = raw"$\rm p_c=0.5/a$"


LoadError: UndefVarError: `load_nk_times` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [4]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
fig.subplots_adjust(wspace=0.3)

ax[1].plot(k, bg_occ[sidx[1], :], "lightgrey", label=raw"$\rm t="*string(Int(round(snapshot_actual_times[1])))*raw"$")
ax[1].plot(k, bg_occ[sidx[2], :], "r", label=raw"$\rm t="*string(Int(round(snapshot_actual_times[2])))*raw"\hbar/\gamma$")
ax[1].plot(k, bg_occ[sidx[3], :], "k", label=raw"$\rm t="*string(Int(round(snapshot_actual_times[3])))*raw"\hbar/\gamma$")

ax_sse = ax[2].twinx()

ax[2].plot(bg_t, bg_teff, "r")
ax[2].plot(fermi_t, fermi_teff, "b")
ax_sse.plot(bg_t, bg_sse, "r--")
ax_sse.plot(fermi_t, fermi_sse, "b--")

ax[1].set_xlabel(raw"$\rm Wavevector \ ka$", fontsize=fs)
ax[1].set_ylabel(raw"$\rm Occupation \ n_k(t)$", fontsize=fs)
ax[1].set_xlim(-pi, pi)
ax[1].set_ylim(0, 1.2)
ax[1].set_xticks([-pi, -pi/2, 0.0, pi/2, pi])
ax[1].set_xticklabels([raw"$-\pi$", raw"$-\pi/2$", raw"$0$", raw"$\pi/2$", raw"$\pi$"])
ax[1].legend(frameon=false, loc="lower center", handlelength=0.5, handletextpad=0.5)
ax[1].text(x=0.12, y=0.9, s=bg_eta_label, transform=ax[1].transAxes, fontsize=fs)

ax[2].set_xlabel(raw"$\rm Time \ \hbar/\gamma$", fontsize=fs)
ax[2].set_ylabel(raw"$\rm T_{eff}(t)$", fontsize=fs)
ax[2].set_xlim(0, 60)
ax[2].set_ylim(-0.1, 1.9)

ax[2].text(x=0.2, y=0.3, s=bg_eta_label_1, transform=ax[2].transAxes, fontsize=fs, color="r")
ax[2].text(x=0.2, y=0.2, s=bg_eta_label_2, transform=ax[2].transAxes, fontsize=fs, color="r")
ax[2].text(x=0.6, y=0.8, s=fermi_eta_label_1, transform=ax[2].transAxes, fontsize=fs, color="b")
ax[2].text(x=0.6, y=0.7, s=fermi_eta_label_2, transform=ax[2].transAxes, fontsize=fs, color="b")

ax_sse.set_ylabel(raw"$\rm \sum \Delta n_k(t)^2$", fontsize=fs, rotation=-90, labelpad=30)

for (i, a) in enumerate(ax)
    a.text(x=0.01, y=0.9, s=letter[i], transform=a.transAxes, fontsize=fs)
end

fig.align_labels()
# Save beside the notebook; on machines without LaTeX, mathtext is used automatically.
fig.savefig(joinpath(repo_root, "Figures_electron_bath", "chain_dissipative_repo.pdf"), bbox_inches="tight")
fig


LoadError: UndefVarError: `plt` not defined in `Main`
Suggestion: check for spelling errors or missing imports.